# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset ([Croissant schema here](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)) using the `mlcroissant` library. You will learn how to query the schema, extract records, and conduct basic data analysis.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for the FAIR^2 dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets (tables), fields, and their unique `@id`s.

Below, we enumerate the record sets and their fields, all by their `@id`.

In [ ]:
# List all record sets (@id, name)
record_sets = list(dataset.record_sets)
if not record_sets:
    print('No record sets found in the schema!')
else:
    print("Record sets found:")
    for rs in record_sets:
        print(f"- @id: {rs['@id']} | Name: {rs.get('name', '[no name]')}")

# For illustration, print the fields (columns/@id) for each record set
for rs in record_sets:
    print(f"\nFields for Record Set: {rs.get('name','[no name]')} (@id: {rs['@id']})")
    fields = rs.get('field', [])
    # Guarantee fields is a list
    if isinstance(fields, dict):
        fields = [fields]
    elif not isinstance(fields, list):
        fields = []
    for fd in fields:
        # Field can be dict or @id string, resolve accordingly
        if isinstance(fd, dict):
            print(f"  - @id: {fd.get('@id')} | Name: {fd.get('name','[no name]')}")
        elif isinstance(fd, str):
            print(f"  - @id: {fd}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Record set and field `@id`s should match those from the overview above.

Below, we extract all record sets into DataFrames, using their `@id`s.

In [ ]:
# Extract data from all available record sets
# Get all record set @id's
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    try:
        # The mlcroissant API expects record_set @id
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(dataframes[record_set_id])} records from record set @id: {record_set_id}")
    except Exception as e:
        print(f"Could not load {record_set_id}: {e}")

# Show the columns for the main tabular record set (choose the largest set)
if dataframes:
    # Try to identify the main set (usually the biggest)
    main_record_set_id = max(dataframes, key=lambda k: len(dataframes[k].columns))
    print(f"\nPrimary record set (most columns): {main_record_set_id}")
    print("Columns (@id):", dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print('No dataframes loaded!')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes to prepare for further analysis.

*All columns and record sets referenced by their `@id`*.

In [ ]:
# Choose a record set and numeric field by @id for basic EDA
# Use the main record set identified before
record_set_id = main_record_set_id
df = dataframes[record_set_id]
print(f"Operating on record set @id: {record_set_id}")

# Let's guess at a numeric field: Try common identifiers
import re
possible_numeric_fields = [col for col in df.columns if re.search(r'age|interval|duration|count|size|score|number', col, re.I)]
if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]
else:
    # fallback: select first column with dtype float or int
    numeric_fields_infer = df.select_dtypes(include=['float', 'int']).columns
    if len(numeric_fields_infer)>0:
        numeric_field_id = numeric_fields_infer[0]
    else:
        numeric_field_id = df.columns[0]
print(f"Selected numeric field @id: {numeric_field_id}")

# Try filtering: values greater than sample threshold (customize as needed)
threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
filtered_df = df[df[numeric_field_id]>threshold] if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else df
print(f"Filtered records with {numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Normalize numeric variable
if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id]-filtered_df[numeric_field_id].mean())/filtered_df[numeric_field_id].std()
    print(f"Normalized field '{numeric_field_id}_normalized':")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping/categorizing by a non-numeric field
categorical_fields = [col for col in df.columns if col != numeric_field_id and pd.api.types.is_object_dtype(df[col])]

if categorical_fields:
    group_field = categorical_fields[0]
    if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field}:")
        display(grouped_df)
else:
    print("No categorical grouping field found in dataframe.")

## 5. Visualization
Visualize the distribution of a numeric field, or the relationship to a grouped/categorical field if available.

In [ ]:
# Simple matplotlib/seaborn visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

# Boxplot per group if group_field defined
if 'group_field' in locals() and pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    plt.figure(figsize=(8,4))
    sns.boxplot(x=group_field, y=numeric_field_id, data=filtered_df)
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to use the `mlcroissant` library to query a FAIR dataset via its Croissant schema. We listed record sets and fields, extracted tabular data by `@id`, applied simple filtering and normalization, grouped by categorical variables, and created basic visualizations.

For a more tailored exploration, review available `@id`s (see section 2) and adapt the analysis to your particular research questions.